# HIT140 Foundations of Data Science — Assignment 2
## Positional Shooting Precision
**Analytic question:** Do Forwards record a significantly higher average of Shots on Target per 90 minutes than Midfielders?

Run the notebooks in numerical order. Each notebook reads the CSV exported by the preceding stage, so the workflow remains reproducible.

# Notebook 2 — Data Preparation and Sampling
This stage examines the cleaned population, checks potential outliers using the IQR rule, visualises the data, and draws a reproducible stratified sample of 64 Forwards and 64 Midfielders.

**Important:** potential outliers are flagged, not automatically deleted. Extreme SoT/90 values can be legitimate football performances, so removing them without a substantive reason could bias the analysis.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

input_file = "01_wrangled_player_data.csv"
if not os.path.exists(input_file):
    raise FileNotFoundError("Run Notebook 1 first to create the cleaned population CSV.")

df_clean = pd.read_csv(input_file)

print("Cleaned population shape:", df_clean.shape)
print(df_clean["Position_Group"].value_counts())
display(df_clean.head())


In [ ]:
# Check potential SoT/90 outliers separately within each position group using the IQR rule.
def flag_iqr_outliers(group):
    q1 = group["SoT/90"].quantile(0.25)
    q3 = group["SoT/90"].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    result = group.copy()
    result["IQR_Lower_Bound"] = lower
    result["IQR_Upper_Bound"] = upper
    result["Potential_Outlier"] = (
        (result["SoT/90"] < lower) | (result["SoT/90"] > upper)
    )
    return result

df_prepared = (
    df_clean.groupby("Position_Group", group_keys=False)
            .apply(flag_iqr_outliers)
            .reset_index(drop=True)
)

outlier_summary = (
    df_prepared.groupby("Position_Group")["Potential_Outlier"]
               .agg(["sum", "count"])
               .rename(columns={"sum":"Potential_Outliers", "count":"Total_Players"})
)
outlier_summary["Percent_Outliers"] = (
    100 * outlier_summary["Potential_Outliers"] / outlier_summary["Total_Players"]
)

print("Potential SoT/90 outliers using the 1.5 × IQR rule:")
display(outlier_summary)

print("\nPotential outlier records:")
display(
    df_prepared.loc[df_prepared["Potential_Outlier"],
                    ["Player_ID","Player","Position_Group","90s","SoT/90",
                     "IQR_Lower_Bound","IQR_Upper_Bound"]]
    .sort_values(["Position_Group","SoT/90"], ascending=[True,False])
)


In [ ]:
# Boxplot: useful for seeing median, spread and potential extreme values.
plt.figure(figsize=(8, 5))
data_to_plot = [
    df_prepared.loc[df_prepared["Position_Group"] == "Forward", "SoT/90"],
    df_prepared.loc[df_prepared["Position_Group"] == "Midfielder", "SoT/90"]
]
plt.boxplot(data_to_plot, tick_labels=["Forward", "Midfielder"])
plt.title("SoT/90 by Position Group — Cleaned Population")
plt.xlabel("Position Group")
plt.ylabel("Shots on Target per 90 (SoT/90)")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("02_population_sot90_boxplot.png", dpi=150)
plt.show()


In [ ]:
# Scatterplot: examine whether SoT/90 behaves differently for players with different playing time.
plt.figure(figsize=(9, 6))

for position in ["Forward", "Midfielder"]:
    subset = df_prepared[df_prepared["Position_Group"] == position]
    plt.scatter(subset["90s"], subset["SoT/90"], alpha=0.6, label=position)

plt.title("Playing Time vs Shots on Target per 90")
plt.xlabel("Equivalent 90-Minute Periods (90s)")
plt.ylabel("Shots on Target per 90 (SoT/90)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("02_90s_vs_sot90_scatterplot.png", dpi=150)
plt.show()


In [ ]:
# Population histograms before sampling.
plt.figure(figsize=(10, 5))

for position in ["Forward", "Midfielder"]:
    subset = df_prepared[df_prepared["Position_Group"] == position]["SoT/90"]
    plt.hist(subset, bins=15, alpha=0.5, label=position)

plt.title("Population Distribution of SoT/90")
plt.xlabel("Shots on Target per 90 (SoT/90)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.savefig("02_population_sot90_histogram.png", dpi=150)
plt.show()


In [ ]:
# Reproducible stratified random sampling.
# We sample the same number from each position because the research question directly compares the two groups.
SAMPLE_SIZE_PER_GROUP = 64
RANDOM_STATE = 42

forwards = df_prepared[df_prepared["Position_Group"] == "Forward"]
midfielders = df_prepared[df_prepared["Position_Group"] == "Midfielder"]

if len(forwards) < SAMPLE_SIZE_PER_GROUP or len(midfielders) < SAMPLE_SIZE_PER_GROUP:
    raise ValueError("One position group contains fewer than 64 eligible players.")

forward_sample = forwards.sample(n=SAMPLE_SIZE_PER_GROUP, random_state=RANDOM_STATE)
midfielder_sample = midfielders.sample(n=SAMPLE_SIZE_PER_GROUP, random_state=RANDOM_STATE)

sample_df = pd.concat([forward_sample, midfielder_sample], ignore_index=True)

print("Sample size by position:")
print(sample_df["Position_Group"].value_counts())
print("\nTotal sample size:", len(sample_df))

# Check how many potential IQR outliers happened to enter the random sample.
print("\nPotential outliers in the selected sample:")
print(sample_df.groupby("Position_Group")["Potential_Outlier"].sum())


In [ ]:
# Export files used by subsequent notebooks.
df_prepared.to_csv("02_prepared_population_with_outlier_flags.csv",
                   index=False, encoding="utf-8-sig")
sample_df.to_csv("02_sampled_player_data.csv",
                 index=False, encoding="utf-8-sig")
outlier_summary.to_csv("02_outlier_summary.csv", encoding="utf-8-sig")

print("Exported:")
print(" - 02_prepared_population_with_outlier_flags.csv")
print(" - 02_sampled_player_data.csv")
print(" - 02_outlier_summary.csv")
